# 01 · Data preparation

Builds a **manifest**: one row per image with its split, file path and all 45 disease labels.

**What changed from the original Part 1 and Part 4 notebooks**
- Images are no longer moved into `Yes_disease/`, `No_disease/` or disease folders. Moving files lost labels: an image with both DR and ODC could only live in one folder, and about 150 DR/MH/ODC images ended up in other folders and were silently dropped.
- The manifest keeps every label, so the disease model can be multi-label.
- Each image is cropped to the fundus and shrunk once to 512 px. This turns about 40 GB of PNGs into under 1 GB and makes training on a Colab GPU practical.
- Missing or duplicate images raise an error instead of being skipped silently.

Runtime: about 15 to 25 minutes the first time (mostly caching), seconds after that.

In [ ]:
# --- Setup (run once per session) -------------------------------------------
# On Colab: mounts Google Drive, clones this repo and installs requirements.
# Locally: just makes the `iluminado` package importable.
import os, sys, subprocess

REPO_URL = "https://github.com/cathykam2006/Capstone-Project-Illuminado-Retinal--Image-Classification.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

REPO_DIR = next(p for p in [os.getcwd(), os.path.dirname(os.getcwd())] if os.path.isdir(os.path.join(p, "iluminado")))
sys.path.insert(0, REPO_DIR)
print("Repo:", REPO_DIR)

In [ ]:
# --- Config: edit these paths ---------------------------------------------------
# PROJECT_DIR holds the cached images, trained models and results. On Colab it
# lives on your Google Drive so it survives between sessions.
PROJECT_DIR = "/content/drive/MyDrive/iluminado" if IN_COLAB else os.path.join(REPO_DIR, "work")
LABEL_DIR = os.path.join(REPO_DIR, "data")            # the three RFMiD label CSVs (in the repo)
MANIFEST = os.path.join(PROJECT_DIR, "manifest.csv")  # written by notebook 01
os.makedirs(PROJECT_DIR, exist_ok=True)

## Get the images

Pick **one** option.

- **Option A (default): download from Kaggle with `kagglehub`.** No 40 GB upload needed. The first run asks for your Kaggle API token. If the dataset slug has changed, search Kaggle for "RFMiD" and update `KAGGLE_SLUG`.
- **Option B: you already have the images.** Set `IMAGE_ROOT` to the folder that contains the training, validation and test image folders (for example on your Drive) and set `USE_KAGGLE = False`.

The loader finds images by folder name (anything containing *train*, *valid*/*evaluation*, *test*), so either layout works.

In [ ]:
USE_KAGGLE = True
KAGGLE_SLUG = "andrewmvd/retinal-disease-classification"
IMAGE_ROOT = "/content/drive/MyDrive/RFMiD"   # only used if USE_KAGGLE = False

if USE_KAGGLE:
    import kagglehub
    IMAGE_ROOT = kagglehub.dataset_download(KAGGLE_SLUG)
print("Images under:", IMAGE_ROOT)

## Build and check the manifest

In [ ]:
import pandas as pd
from iluminado.data import build_manifest, TARGET_DISEASES

manifest = build_manifest(IMAGE_ROOT, LABEL_DIR)

expected = {"train": 1920, "val": 640, "test": 640}
counts = manifest["split"].value_counts().to_dict()
assert counts == expected, f"Unexpected split sizes: {counts}"
print("All 3,200 labelled images found:", counts)
manifest.head()

## Why multi-label

How many images carry more than one of the three target diseases, and how many target-disease images would the old folder approach have dropped?

In [ ]:
t = manifest[TARGET_DISEASES]
summary = pd.DataFrame({
    "images with DR/MH/ODC": manifest.assign(k=t.sum(axis=1)).query("k >= 1").groupby("split").size(),
    "with 2+ of them": manifest.assign(k=t.sum(axis=1)).query("k >= 2").groupby("split").size(),
    "with any 2+ diseases": manifest.assign(k=manifest.drop(columns=["split", "ID", "path", "Disease_Risk"]).sum(axis=1)).query("k >= 2").groupby("split").size(),
})
summary

## Crop and cache

In [ ]:
from iluminado.data import cache_images

CACHE_SIZE = 512   # training uses 384 by default; caching a bit larger leaves room to experiment
manifest_raw_path = os.path.join(PROJECT_DIR, "manifest_raw.csv")
manifest.to_csv(manifest_raw_path, index=False)   # original paths, used by the EDA notebook

cached = cache_images(manifest, os.path.join(PROJECT_DIR, f"cache_{CACHE_SIZE}"), size=CACHE_SIZE)
cached.to_csv(MANIFEST, index=False)
print("Saved", MANIFEST, len(cached), "rows")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes[1], cached.head(4).iterrows()):
    ax.imshow(Image.open(row.path)); ax.set_title(f"cached #{row.ID}"); ax.axis("off")
for ax, (_, raw) in zip(axes[0], manifest.head(4).iterrows()):
    ax.imshow(Image.open(raw.path)); ax.set_title(f"original #{raw.ID}"); ax.axis("off")
plt.tight_layout(); plt.show()